# SHIPP Stage 8A — Silver listing files

`shipp_bronze.lb_listing_files_history` (Lakebase CDC) + files in the `listing_images` Volume → `shipp_silver.silver_listing_files`

**Grain:** one row per current `listing_file_id` whose listing still exists.
**Delete-aware:** same `reconstruct_current_state` rule as Silver listings/requests.
**File truth:** `file_exists`, `file_size_bytes`, `file_sha256` come from actually reading the Volume, not from Lakebase.
Lakebase stores only the path — never image bytes.

**Clear state and outputs → Run all.** Every gate prints PASS or stops. Never loosen an assert. Only writer of `silver_listing_files`.

In [ ]:
%run ../common/shipp_silver_lib

In [ ]:
# Reload the rag modules so edits to rag/*.py always take effect (same rule as shipp_silver_lib).
for _name in ["rag.text", "rag.vision", "rag.content", "rag.search_docs", "rag.index",
              "rag.retrieval", "rag.semantic", "rag.evaluation"]:
    importlib.reload(importlib.import_module(_name))

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
print("RAG modules reloaded from", REPO_ROOT)

## Step 1 — Config

In [ ]:
from rag import content

SOURCE_TABLE = BRONZE_LISTING_FILES_HISTORY
TARGET_TABLE = SILVER_LISTING_FILES
BUSINESS_KEY = "listing_file_id"
REQUIRED_SOURCE_COLUMNS = {"listing_file_id", "listing_id", "file_path", "file_type", "uploaded_at",
                           "_pg_change_type", "_pg_lsn", "_sort_by"}

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {LISTING_IMAGES_VOLUME_NAME}")
print("SOURCE_TABLE =", SOURCE_TABLE)
print("TARGET_TABLE =", TARGET_TABLE)
print("VOLUME       =", LISTING_IMAGES_VOLUME)
assert table_exists(SOURCE_TABLE), (
    f"FAILED: {SOURCE_TABLE} does not exist. Sync shipp.listing_files from Lakebase exactly like "
    "lb_listings_history (rag/README.md → Setup step 2)."
)

## Step 2 — Load Bronze (pinned) + source schema contract

In [ ]:
bronze_df, bronze_version = read_bronze_pinned(SOURCE_TABLE)
print("Bronze row count:", bronze_df.count())
missing_columns = REQUIRED_SOURCE_COLUMNS - set(bronze_df.columns)
assert not missing_columns, f"Bronze schema contract failed. Missing: {sorted(missing_columns)}"
print("PASS — Bronze source schema contract.")
display(bronze_df.groupBy("_pg_change_type").count().orderBy("_pg_change_type"))

## Step 3 — Delete-aware current state + keep only files of current listings

In [ ]:
latest_events_df, current_rows_df, deleted_keys_df = reconstruct_current_state(bronze_df, BUSINESS_KEY)
current_listings = spark.table(SILVER_LISTINGS).select("listing_id")
orphans = current_rows_df.join(current_listings, "listing_id", "left_anti").count()

print("File keys with any event:   ", latest_events_df.count())
print("Newest event = delete (out):", deleted_keys_df.count())
print("Current file rows:          ", current_rows_df.count())
print("Orphans (listing not in Silver) excluded:", orphans)

## Step 4 — Read the real files in the Volume (runbook 8A.2)

In [ ]:
try:
    binary_df = (spark.read.format("binaryFile")
                 .option("recursiveFileLookup", "true")
                 .load(LISTING_IMAGES_VOLUME))
    volume_facts = content.volume_file_facts(binary_df)
    volume_count = volume_facts.count()
except Exception as e:
    print(f"WARNING — could not read {LISTING_IMAGES_VOLUME}: {type(e).__name__}: {str(e)[:200]}")
    volume_facts = spark.createDataFrame([], "file_path string, file_size_bytes long, file_sha256 string")
    volume_count = 0
print("Files found in the Volume:", volume_count)
display(volume_facts.orderBy("file_path"))

## Step 5 — Build Silver + quality gate

In [ ]:
silver_df = content.build_silver_listing_files(current_rows_df, current_listings, volume_facts)
row_count = silver_df.count()

run_quality_gate({
    "duplicate listing_file_id": duplicate_key_count(silver_df, BUSINESS_KEY),
    "null listing_id": null_count(silver_df, "listing_id"),
    "null file_path": null_count(silver_df, "file_path"),
    "file_path outside listing_images volume": silver_df.filter(
        ~F.col("file_path").startswith(LISTING_IMAGES_VOLUME + "/")).count(),
    "file exists but is empty": silver_df.filter(F.col("file_exists") & (F.col("file_size_bytes") <= 0)).count(),
    "delete event in Silver": silver_df.filter(~F.col("source_change_type").isin(SILVER_SOURCE_CHANGE_TYPES)).count(),
}, row_count=row_count, allow_empty=True)

missing_files = silver_df.filter(~F.col("file_exists")).count()
readable_images = silver_df.filter(F.col("file_exists") & content.is_image_path(F.col("file_path"))).count()
if missing_files:
    print(f"WARNING — {missing_files} file row(s) point to a path that is not in the Volume (→ FILE_MISSING).")
print("8A.2 PASS — at least one real image exists and is readable." if readable_images
      else "8A.2 NOT MET — upload at least one image to the Volume (rag/README.md → Setup step 3).")
display(silver_df.orderBy("listing_id", "listing_file_id"))

## Step 6 — Write + exact schema + idempotency

In [ ]:
written = write_overwrite(silver_df, TARGET_TABLE)
assert written == row_count, f"WRITE VALIDATION FAILED: wrote {written}, validated {row_count}"
assert_exact_schema(TARGET_TABLE, content.LISTING_FILE_COLUMNS)

first = logical_snapshot(TARGET_TABLE, BUSINESS_KEY)
write_overwrite(content.build_silver_listing_files(current_rows_df, current_listings, volume_facts), TARGET_TABLE)
assert logical_snapshot(TARGET_TABLE, BUSINESS_KEY) == first, "FAILED: rerun changed the table (not idempotent)"
print("PASS — rerun produced the same table.")

In [ ]:
summary = {"table": TARGET_TABLE, "rows": written, "missing_files": missing_files,
           "readable_images": readable_images, "bronze_version": bronze_version, "status": "PASS"}
print(summary)
dbutils.notebook.exit(json.dumps(summary, default=str))